# LAB | API Calling to ChatGPT

#### Ironhack - AI Integration & Consulting


Scenario You're working as an AI automation specialist for an e-commerce company. Your team needs to automate the creation of product listings from product images and basic information. Instead of manually writing descriptions, you'll use ChatGPT's vision capabilities to generate compelling product listings automatically. This will save hours of manual work and ensure consistent, high-quality product descriptions.

### Step 1: Setting Up OpenAI API Access

Objective: Get API access and set up authentication.

Done during class 

## Step 2: Preparing the Dataset

Objective: Download and set up the e-commerce product dataset.

In [1]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

c:\Users\antun\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [4]:
# Inspect what was actually loaded
print(products_df.columns.tolist())
print(products_df.iloc[0].to_dict())

# Checking if image has a file path
print(products_df.columns.tolist())
print(products_df.head(1).to_dict())
print(type(products_df.iloc[0]["image"]))

['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
{'id': 15970, 'gender': 'Men', 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'season': 'Fall', 'year': 2011.0, 'usage': 'Casual', 'productDisplayName': 'Turtle Check Men Navy Blue Shirt', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=60x80 at 0x2BFEB36DAD0>}
['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
{'id': {0: 15970}, 'gender': {0: 'Men'}, 'masterCategory': {0: 'Apparel'}, 'subCategory': {0: 'Topwear'}, 'articleType': {0: 'Shirts'}, 'baseColour': {0: 'Navy Blue'}, 'season': {0: 'Fall'}, 'year': {0: 2011.0}, 'usage': {0: 'Casual'}, 'productDisplayName': {0: 'Turtle Check Men Navy Blue Shirt'}, 'image': {0: <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=60x80 at 0x2BFEB36DAD

## Step 3: Encoding Images for API


Objective: Convert product images to base64 format for API transmission.

In [5]:
# Encond the image memory, based on the result <class 'PIL.JpegImagePlugin.JpegImageFile'>

import base64
from io import BytesIO

def encode_pil_image_to_base64(image):
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

sample_image = products_df.iloc[0]["image"]

encoded_image = encode_pil_image_to_base64(sample_image)

print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


## Step 4: Creating the Product Listing Prompt

Objective: Design an effective prompt for generating product listings.

In [6]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


## Step 5: Calling the ChatGPT API with Vision

Objective: Send image and text to ChatGPT API and receive response.

What to do:

- Prepare the API request with image and prompt
- Call the ChatGPT API
- Handle the response
- Parse JSON output

In [7]:
# Install OpenAI package
%pip install -U openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from openai import OpenAI

In [14]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
import json
from openai import OpenAI

# Prompt created in Step 4
prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

# Image encoded in Step 3
image_data_url = f"data:image/jpeg;base64,{encoded_image}"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url
                    }
                }
            ]
        }
    ],
    
)

raw_result = response.choices[0].message.content
print(raw_result) # Printing raw_result first to see the exact format returned and identify why the response might fail.

```json
{
    "title": "Ultimate Wireless Bluetooth Headphones with Noise Cancelling",
    "description": "Experience unparalleled sound quality with our Ultimate Wireless Bluetooth Headphones, perfect for music lovers and audio professionals alike. Priced at just $79.99, these headphones are designed to block out distractions with advanced noise-cancelling technology, ensuring you can immerse yourself in your favorite tunes or podcasts without interruption. With a staggering 30-hour battery life, enjoy extended listening sessions whether you're at home, commuting, or on a workout spree. The sleek and ergonomic design not only enhances comfort but also ensures a secure fit. Ideal for all-day wear, these headphones come in a stylish finish that complements any outfit. Elevate your audio experience today and rediscover your playlists like never before!",
    "features": [
        "Advanced Noise Cancelling Technology",
        "30-Hour Battery Life for All-Day Use",
        "Wireless Blu

## Step 6: Processing Multiple Products

Objective: Generate listings for multiple products in batch.

What to do:

- Loop through products
- Generate listing for each
- Save results
- Handle errors gracefully

In [11]:
print(products_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  100 non-null    int64  
 1   gender              100 non-null    str    
 2   masterCategory      100 non-null    str    
 3   subCategory         100 non-null    str    
 4   articleType         100 non-null    str    
 5   baseColour          100 non-null    str    
 6   season              100 non-null    str    
 7   year                100 non-null    float64
 8   usage               100 non-null    str    
 9   productDisplayName  100 non-null    str    
 10  image               100 non-null    object 
dtypes: float64(1), int64(1), object(1), str(8)
memory usage: 16.4+ KB
None


In [12]:
def create_product_listing_prompt(productDisplayName, year, masterCategory):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {productDisplayName}
- Year: {year}
- Category: {masterCategory}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt

 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [13]:
all_results = []

for index, product in products_df.iterrows():
    prompt = create_product_listing_prompt(
        productDisplayName = product["productDisplayName"],
        year = product["year"],
        masterCategory = product["masterCategory"],
        )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_data_url
                        }
                    }
                ]
            }
        ],
        
    )


    content = response.choices[0].message.content

    print(f"\n{index}:")
    print(f"\t{content}")

    all_results.append({content: content})

print(all_results)


0:
	```json
{
    "title": "Stylish Navy Blue Turtle Check Shirt for Men",
    "description": "Upgrade your wardrobe with the Turtle Check Men Navy Blue Shirt, a perfect blend of style and comfort. Crafted from high-quality fabric, this shirt offers breathability and a soft touch, making it ideal for both casual outings and semi-formal occasions. The navy blue shade is versatile, allowing you to pair it effortlessly with jeans or chinos. Featuring a classic check pattern, this shirt adds a touch of modern sophistication to your ensemble. The slightly rolled sleeves provide a laid-back vibe, while the button-down collar ensures a polished look. With its rich design elements and exceptional fit, this shirt is perfect for the modern man who values style as much as comfort.",
    "features": [
        "High-quality breathable fabric",
        "Stylish navy blue color",
        "Classic check pattern for a trendy look",
        "Button-down collar for a polished appearance",
        "Sligh

## Report 

- How the API integration works?

The application automates product-listing creation. It first loads the product information, then each image is read and encoded so it can be included in the API request.

- Challenges you faced

One challenge was ensuring that the API returned valid JSON rather than plain text or Markdown code blocks. I addressed this by checking the raw response. Another challenge was that the model sometimes generated assumptions about products. I fixed that by double checking my dataset and making sure the model got the correct input.

- Quality of generated listings

From what I understand the quality of the generated listings were good - readable, consistent, and suitable. However a further investigation could be done in regards to details on the images. Overall, I believe the model can automate the first draft of e-commerce product listings well. 

- Potential improvements

For what was requested to this LAB, I believe the model is very satisfying. The integration successfully combined images, metadata, prompts, and API responses into a repeatable workflow. There are possibility room for improvement yes, specially if we think about certain details of the product. 